# Extra da Aula 02 · Fundamentos de LangGraph

**Do Hello World à execução de um grafo.** Primeiro, uma função recebe um nome e devolve uma saudação. Depois, duas funções cooperam para medir e classificar a taxa de falhas de uma janela fictícia de transferências.

Ao final, você consegue apontar o **estado**, os **nós** e as **arestas**, montar o fluxo e executá-lo com `invoke()`. No fim, você escolhe um caminho com uma aresta condicional e usa a OpenAI para redigir a mensagem de um alerta fictício.

## Preparação

**Antes de rodar**, abra o notebook no VS Code com as extensões Python e Jupyter. Escolha um dos caminhos da tabela.

| Situação | O que fazer |
|---|---|
| **Já tem a venv da Aula 02** | Em **Selecionar kernel → Ambientes Python**, escolha `projeto/.venv/Scripts/python.exe` no Windows ou `projeto/.venv/bin/python` no macOS ou Linux. Vá direto ao Hello World. |
| **Só tem este notebook · Windows** | No PowerShell, na pasta do notebook, rode em ordem:<br>**1.** `py -3.12 -m venv .venv`<br>**2.** `.\.venv\Scripts\python.exe -m pip install langgraph==1.2.19 ipykernel==7.4.0`<br>**3.** Selecione o kernel `.venv/Scripts/python.exe`. |
| **Só tem este notebook · macOS ou Linux** | No terminal, na pasta do notebook, rode em ordem:<br>**1.** `python3.12 -m venv .venv`<br>**2.** `./.venv/bin/python -m pip install langgraph==1.2.19 ipykernel==7.4.0`<br>**3.** Selecione o kernel `.venv/bin/python`. |

Com o kernel selecionado, execute as células de cima para baixo. A importação de LangGraph aparece na montagem do Hello World.

### LangChain, LangGraph e LangSmith

| Ferramenta | Quando entra | Documentação oficial |
|---|---|---|
| **LangChain** | Quando você quer conectar modelos e ferramentas ou montar um agente com componentes prontos. | [Visão geral do LangChain](https://docs.langchain.com/oss/python/langchain/overview) |
| **LangGraph** | Quando você quer desenhar o fluxo, definir o estado e escolher como as funções se conectam. É o que vamos usar aqui. | [Visão geral do LangGraph](https://docs.langchain.com/oss/python/langgraph/overview) · [Graph API em Python](https://docs.langchain.com/oss/python/langgraph/graph-api) |
| **LangSmith** | Quando você quer inspecionar as etapas de uma execução ou avaliar os resultados de uma aplicação. | [Documentação do LangSmith](https://docs.langchain.com/langsmith/observability) |

## 1 · Hello World. Teste a função antes de montar o grafo

Vamos receber um dicionário com `nome` e devolver outro dicionário com `mensagem`. O primeiro representa o **estado** que a função lê. O segundo representa a **atualização** que ela quer fazer no estado.

Usamos `TypedDict` para descrever os campos do estado. `nome` está disponível na entrada e `mensagem` pode aparecer depois, por isso a anotamos como `NotRequired`. Quando registrarmos a função no grafo, ela será um **nó**.

In [ ]:
from typing import NotRequired, TypedDict


class EstadoOla(TypedDict):
    nome: str
    mensagem: NotRequired[str]


def saudar(estado: EstadoOla) -> dict[str, str]:
    return {"mensagem": f"Olá, {estado['nome']}!"}


entrada_ola = {"nome": "turma"}
print(saudar(entrada_ola))

A função devolveu `{'mensagem': 'Olá, turma!'}`. Agora vamos conferir o que muda quando quem chama a função é o LangGraph.

## 2 · Registre o nó, conecte as setas e execute

```text
START ──► saudar ──► END
```

| Linha | O que ela faz |
|---|---|
| `StateGraph(EstadoOla)` | Diz quais campos o fluxo pode ler e atualizar. |
| `add_node("saudar", saudar)` | Associa o nome do nó à função Python. |
| `add_edge(START, "saudar")` | Indica qual nó roda primeiro. |
| `add_edge("saudar", END)` | Encerra o fluxo depois da saudação. |
| `compile()` | Prepara o grafo para ser executado. |
| `invoke(entrada_ola)` | Roda o grafo com a entrada e devolve o estado final. |

In [ ]:
from langgraph.graph import END, START, StateGraph

montagem_ola = StateGraph(EstadoOla)
montagem_ola.add_node("saudar", saudar)
montagem_ola.add_edge(START, "saudar")
montagem_ola.add_edge("saudar", END)

grafo_ola = montagem_ola.compile()
estado_final_ola = grafo_ola.invoke(entrada_ola)
print(estado_final_ola)

A saída é `{'nome': 'turma', 'mensagem': 'Olá, turma!'}`. Na chamada direta, vimos apenas a atualização que a função devolveu. Com `invoke()`, vemos o estado completo ao final da execução. O grafo guardou `nome` e acrescentou `mensagem`.

## 3 · Faça dois nós trabalharem juntos

Agora imagine uma janela fictícia de transferências Pix. Recebemos o total de solicitações e o total de falhas. O primeiro nó calcula a taxa. O segundo lê essa taxa e aplica um limiar didático de 1%.

```text
START ──► calcular_taxa ──► classificar_janela ──► END
```

| Nó | Lê | Acrescenta ao estado |
|---|---|---|
| `calcular_taxa` | `solicitacoes`, `falhas` | `taxa_erro` |
| `classificar_janela` | `taxa_erro` | `decisao` |

Repare que `classificar_janela()` espera a chave `taxa_erro`. A aresta garante a ordem: quando o segundo nó começar, o primeiro já terá escrito esse campo. Se o total for zero, a taxa fica indefinida e a decisão indica dados insuficientes.

In [ ]:
class EstadoJanela(TypedDict):
    solicitacoes: int
    falhas: int
    taxa_erro: NotRequired[float | None]
    decisao: NotRequired[str]


def calcular_taxa(estado: EstadoJanela) -> dict[str, float | None]:
    total = estado["solicitacoes"]
    taxa = estado["falhas"] / total if total > 0 else None
    return {"taxa_erro": taxa}


def classificar_janela(estado: EstadoJanela) -> dict[str, str]:
    taxa = estado["taxa_erro"]
    if taxa is None:
        return {"decisao": "dados_insuficientes"}
    if taxa >= 0.01:
        return {"decisao": "investigar"}
    return {"decisao": "abaixo_do_limiar"}

## 4 · Monte e execute o grafo da janela

O que mudou em relação ao Hello World? Registramos duas funções e acrescentamos uma aresta entre elas. Antes de rodar, tente prever os quatro campos da saída. A entrada tem 1.000 solicitações e 80 falhas.

In [ ]:
montagem_janela = StateGraph(EstadoJanela)
montagem_janela.add_node("calcular_taxa", calcular_taxa)
montagem_janela.add_node("classificar_janela", classificar_janela)
montagem_janela.add_edge(START, "calcular_taxa")
montagem_janela.add_edge("calcular_taxa", "classificar_janela")
montagem_janela.add_edge("classificar_janela", END)

grafo_janela = montagem_janela.compile()

### Veja o desenho antes de executar

O grafo já está compilado. `get_graph()` expõe seus nós e arestas para visualização. A célula abaixo imprime a descrição Mermaid, que você consegue gerar localmente. Para **exibir o diagrama como imagem no notebook**, mude `MOSTRAR_IMAGEM` para `True` e execute a célula. A renderização PNG usa o serviço Mermaid.INK e precisa de internet.

No Hello World, o mesmo comando funciona com `grafo_ola`.

In [ ]:
MOSTRAR_IMAGEM = False

diagrama_janela = grafo_janela.get_graph()
if MOSTRAR_IMAGEM:
    from IPython.display import Image, display

    display(Image(diagrama_janela.draw_mermaid_png()))
else:
    print(diagrama_janela.draw_mermaid())

### Execute com `invoke()`

Agora passe as contagens ao grafo. Confira quais campos os dois nós acrescentaram.

In [ ]:
entrada_janela = {"solicitacoes": 1_000, "falhas": 80}
estado_final_janela = grafo_janela.invoke(entrada_janela)
print(estado_final_janela)

Você deve ver as duas contagens da entrada, `taxa_erro: 0.08` e `decisao: 'investigar'`. A taxa `0.08` representa 8%. Ela foi escrita pelo primeiro nó e lida pelo segundo.

**Experimente na mesma célula.** Troque `"falhas": 80` por `"falhas": 2` e rode novamente. Confira a taxa de 0,2% e a decisão `abaixo_do_limiar`.

No grafo de triagem da Aula 02, a ideia é a mesma: uma função calcula, outra lê o resultado e decide. Se um assistente de IA entrar mais adiante, o grafo permite escolher em qual etapa ele recebe os dados e quais verificações vêm antes dele.

[Documentação oficial da Graph API](https://docs.langchain.com/oss/python/langgraph/graph-api)

## 5 · Aresta condicional. Só monte o alerta quando houver motivo

Até aqui, a sequência era fixa. Agora, depois de `classificar_janela`, queremos escolher o próximo passo com base na decisão que ficou no estado.

```text
START → calcular_taxa → classificar_janela
                          ├─ investigar → montar_alerta → END
                          ├─ abaixo_do_limiar ──────────→ END
                          └─ dados_insuficientes ───────→ END
```

`escolher_caminho()` lê `decisao` e devolve seu valor. O dicionário em `add_conditional_edges()` traduz esse valor no próximo destino. Por exemplo, `"investigar"` aponta para o nó `"montar_alerta"`.

O esquema ganha um campo opcional, `alerta`, que só existe quando esse nó roda.

In [ ]:
class EstadoComAlerta(EstadoJanela):
    alerta: NotRequired[str]


def escolher_caminho(estado: EstadoComAlerta) -> str:
    return estado["decisao"]


def montar_alerta(estado: EstadoComAlerta) -> dict[str, str]:
    return {"alerta": (
        f"Janela Pix fictícia: {estado['falhas']} falhas em "
        f"{estado['solicitacoes']} solicitações. "
        f"Taxa de erro de {estado['taxa_erro']:.1%}."
    )}


montagem_condicional = StateGraph(EstadoComAlerta)
montagem_condicional.add_node("calcular_taxa", calcular_taxa)
montagem_condicional.add_node("classificar_janela", classificar_janela)
montagem_condicional.add_node("montar_alerta", montar_alerta)
montagem_condicional.add_edge(START, "calcular_taxa")
montagem_condicional.add_edge("calcular_taxa", "classificar_janela")
montagem_condicional.add_conditional_edges(
    "classificar_janela",
    escolher_caminho,
    {
        "investigar": "montar_alerta",
        "abaixo_do_limiar": END,
        "dados_insuficientes": END,
    },
)
montagem_condicional.add_edge("montar_alerta", END)
grafo_condicional = montagem_condicional.compile()

In [ ]:
cenarios = {
    "impacto alto": {"solicitacoes": 1_000, "falhas": 80},
    "abaixo do limiar": {"solicitacoes": 1_000, "falhas": 2},
    "sem tráfego": {"solicitacoes": 0, "falhas": 0},
}

resultados = {}
for nome, entrada in cenarios.items():
    estado = grafo_condicional.invoke(entrada)
    resultados[nome] = estado
    print(nome, "→", estado["decisao"], "· alerta:", estado.get("alerta"))

## 6 · Uma chamada direta a um modelo da OpenAI

Só o cenário de **impacto alto** ganhou um `alerta`. Vamos enviar esse texto sintético a um modelo e pedir uma frase para a passagem de plantão. A regra de 1% já tomou a decisão em Python. O modelo só escreve a mensagem.

1. Se o kernel ainda não tem o SDK, rode `%pip install "openai>=2,<3" "python-dotenv>=1,<2"` em uma célula Python. Reinicie o kernel se o editor pedir e rode as células anteriores de novo.
2. Crie um arquivo **`.env` na mesma pasta deste notebook**, com `OPENAI_API_KEY=sua_chave` e, se quiser escolher outro modelo, `OPENAI_MODEL=nome_do_modelo`. Mantenha a chave no arquivo local. O exemplo usa `gpt-4.1-mini` por padrão.
3. Na célula abaixo, mude `CHAMAR_MODELO` para `True` e rode. Isso faz uma chamada real à API e pode gerar custo.

Na saída, confira se a frase mantém os números **80 em 1.000 (8%)** e indica investigação. O alerta informa o impacto observado e não estabelece a causa.

[Documentação oficial da API Responses em Python](https://developers.openai.com/api/reference/python/resources/responses/methods/create)

In [ ]:
CHAMAR_MODELO = False

if CHAMAR_MODELO:
    from pathlib import Path

    from dotenv import dotenv_values
    from openai import OpenAI

    arquivo_env = Path(".env")
    config = dotenv_values(arquivo_env)
    chave = config.get("OPENAI_API_KEY")
    if not chave or chave == "sua_chave":
        raise ValueError(f"Preencha OPENAI_API_KEY no arquivo {arquivo_env.resolve()}.")

    cliente = OpenAI(api_key=chave)
    alerta = resultados["impacto alto"]["alerta"]
    resposta = cliente.responses.create(
        model=config.get("OPENAI_MODEL") or "gpt-4.1-mini",
        instructions=(
            "Escreva uma frase em português para a passagem de plantão. "
            "Mantenha as contagens e a taxa, indique que a janela merece "
            "investigação e não atribua uma causa."
        ),
        input=alerta,
        max_output_tokens=120,
        store=False,
    )
    print(resposta.output_text)
else:
    print("Para chamar a OpenAI, configure o .env e mude CHAMAR_MODELO para True.")